## BERT Encoder Architecture

Welcome back to Harnessing Transformers with Hugging Face! In this second lesson, we're taking a significant leap forward in our exploration of transformer architectures. Having established a solid foundation with the Hugging Face ecosystem in our first lesson—where you mastered pipelines for high-level tasks and `Auto` classes for flexible model loading—we now turn our attention to one of the most groundbreaking innovations in natural language processing.

Today, we dive deep into BERT (Bidirectional Encoder Representations from Transformers), a revolutionary architecture that fundamentally changed how we approach language understanding. While you've mastered the general transformer architecture in previous courses, BERT represents a paradigm shift in how these transformers are applied. Unlike the autoregressive models you've built, which process text sequentially, BERT introduces true bidirectional context understanding—the ability to see both past and future context simultaneously. This architectural choice makes BERT exceptionally powerful for understanding tasks like classification, question answering, and named entity recognition, where comprehending the full context is crucial. By the end of this lesson, you'll understand why BERT became the foundation for countless NLP breakthroughs and how to harness its power through Hugging Face.

---

### Understanding BERT's Revolutionary Architecture

BERT represents a fundamental departure from traditional approaches to language modeling. As you'll recall from building transformers from scratch, most sequence models—including powerful architectures like GPT—process text in a left-to-right manner, seeing only the preceding context when making predictions. This unidirectional approach, while effective for generation tasks, creates a fundamental limitation: the model can't leverage future context that humans naturally use when understanding language.

BERT shatters this limitation through its encoder-only architecture that processes the entire sequence simultaneously. This design choice stems from a brilliant insight: for many NLP tasks, we don't need to generate text—we need to understand it. By using only the encoder stack from the original transformer architecture, BERT can apply self-attention in both directions without the causal masking constraints required for generation. Every word can attend to every other word in the sequence, creating rich, bidirectional representations that capture complex semantic relationships.

The key innovation enabling this bidirectional processing is BERT's unique training objective: masked language modeling (MLM). Instead of predicting the next word in a sequence, BERT randomly masks 15% of input tokens and learns to predict them using the surrounding context. This forces the model to develop deep bidirectional understanding—to predict a masked word, it must integrate information from both before and after that position. Additionally, BERT uses a next sentence prediction task during training, learning whether two sentences naturally follow each other, which helps the model understand relationships between sentence pairs. These training strategies result in a model that excels at understanding tasks where meaning emerges from comprehensive context analysis.

---

### BERT's WordPiece Tokenization Strategy

Before we can leverage BERT's powerful bidirectional understanding, we need to understand how it processes text at the most fundamental level. BERT employs WordPiece tokenization, a sophisticated subword tokenization strategy that elegantly balances vocabulary efficiency with semantic preservation. This approach is crucial for handling the incredible diversity of human language while keeping the model size manageable.

It's also important to note that BERT models have a maximum sequence length determined by their positional embeddings—typically, BERT's maximum position embeddings are set to 512 tokens. This means that after tokenization (including special tokens like `[CLS]` and `[SEP]`), the input sequence cannot exceed 512 tokens. If your input text is longer than this limit, the tokenizer will either truncate the sequence (removing tokens from the end or according to a specified strategy) or, if not handled, the model will raise an error. For tasks involving longer documents, you must split the text into manageable chunks, each fitting within the 512-token constraint, to ensure compatibility with BERT's architecture.

```python
from transformers import AutoTokenizer

def explore_bert_tokenizer():
    """Explore BERT's WordPiece tokenization and sequence length limit"""
    print("BERT WordPiece Tokenization:")

    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    print(f"Maximum sequence length: {tokenizer.model_max_length} tokens\n")

    # Test tokenization scenarios
    test_texts = ["Hello world", "Tokenization", "COVID-19 pandemic"]

    for text in test_texts:
        tokens = tokenizer.tokenize(text)
        token_ids = tokenizer.encode(text, add_special_tokens=True)

        print(f"Text: '{text}'")
        print(f"  Tokens: {tokens}")
        print(f"  Token IDs: {token_ids}")
        print()
```

Running this code reveals the intelligent nature of WordPiece tokenization:

```text
BERT WordPiece Tokenization:
Maximum sequence length: 512 tokens

Text: 'Hello world'
  Tokens: ['hello', 'world']
  Token IDs: [101, 7592, 2088, 102]

Text: 'Tokenization'
  Tokens: ['token', '##ization']
  Token IDs: [101, 19204, 3989, 102]

Text: 'COVID-19 pandemic'
  Tokens: ['co', '##vid', '-', '19', 'pan', '##de', '##mic']
  Token IDs: [101, 2522, 17258, 1011, 2539, 6090, 3207, 7712, 102]
```

The tokenization results reveal WordPiece's intelligent approach to vocabulary management. Common words like `hello` and `world` remain intact as single tokens, ensuring efficient processing for frequent vocabulary. However, less common or complex words undergo subword decomposition: `tokenization` splits into `['token', '##ization']`, where the `##` prefix indicates a subword continuation. This mechanism allows BERT to handle virtually any word by breaking it into known components, even technical terms like `COVID-19`, which decomposes into `['co', '##vid', '-', '19', 'pan', '##de', '##mic']`. The `encode()` method adds special tokens that are crucial for BERT's processing: `[CLS]` (token ID 101) marks the beginning of the sequence and aggregates sequence-level information, while `[SEP]` (token ID 102) marks the end or separates sentence pairs. This tokenization strategy gives BERT remarkable flexibility—it can process any text while maintaining a fixed vocabulary of just 30,522 tokens.

---

### Masked Language Modeling: BERT's Superpower

Now let's witness BERT's bidirectional understanding in action through its signature capability: masked language modeling. This technique demonstrates how BERT leverages both left and right context to make remarkably accurate predictions about missing words—something impossible for traditional left-to-right models.

```python
from transformers import AutoModelForMaskedLM
import torch

def bert_masked_language_modeling():
    """Demonstrate BERT's masked language modeling"""
    print("BERT Masked Language Modeling:")

    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")

    # Test sentences with masks
    test_sentences = [
        "The [MASK] is shining brightly today.",
        "I love to [MASK] books in my free time."
    ]

    for sentence in test_sentences:
        inputs = tokenizer(sentence, return_tensors="pt")

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = outputs.logits

        # Find mask token and get top predictions
        mask_token_index = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)[1]
        mask_logits = predictions[0, mask_token_index, :]
        top_tokens = torch.topk(mask_logits, k=3, dim=1)

        print(f"Sentence: '{sentence}'")
        print("Top predictions:")
        for token_id, score in zip(top_tokens.indices[0], top_tokens.values[0]):
            token = tokenizer.decode([token_id])
            print(f"  '{token.strip()}': score={score.item():.2f}")
        print()
```

The masked language modeling results showcase BERT's sophisticated contextual reasoning:

```text
BERT Masked Language Modeling:
Sentence: 'The [MASK] is shining brightly today.'
Top predictions:
  'sun': score=12.06
  'moon': score=8.84
  'city': score=8.59

Sentence: 'I love to [MASK] books in my free time.'
Top predictions:
  'read': score=14.87
  'write': score=11.08
  'study': score=9.34
```

These results demonstrate BERT's remarkable ability to integrate bidirectional context. For "The `[MASK]` is shining brightly today," BERT confidently predicts `sun` (score: 12.06), demonstrating how it integrates multiple contextual clues: the verb `shining`, the adverb `brightly`, and the temporal reference `today` all point toward celestial objects, with the sun being the most plausible during the daytime. The alternative predictions—`moon` and `city`—show BERT considering other entities that can shine, though with lower confidence.

Similarly, for "I love to `[MASK]` books in my free time," BERT's top prediction is `read` (score: 14.87), showing even higher confidence. The model understands that `books` is a direct object of the masked verb, combined with `love to` and `free time`, which strongly suggests reading as an enjoyable leisure activity. The alternatives `write` and `study` represent other book-related activities, demonstrating BERT's nuanced understanding of different ways people interact with books. These predictions would be impossible for a unidirectional model that couldn't see `books` when predicting the masked verb.

---

### Contextualized Embeddings: Words That Change Meaning

One of BERT's most powerful capabilities is generating contextualized embeddings—word representations that dynamically adapt based on the surrounding context. This represents a quantum leap from traditional word embeddings, where `bank` always has the same representation regardless of whether it refers to a financial institution or a riverbank. Let's explore how BERT creates context-sensitive representations:

```python
from transformers import AutoModel

def bert_contextualized_embeddings():
    """Extract BERT's contextualized embeddings"""
    print("BERT Contextualized Embeddings:")

    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModel.from_pretrained("bert-base-uncased")

    # Test word in different contexts
    contexts = [
        "The bank of the river is muddy.",
        "I need to go to the bank to withdraw money."
    ]

    for i, context in enumerate(contexts):
        inputs = tokenizer(context, return_tensors="pt")
        tokens = tokenizer.tokenize(context)

        with torch.no_grad():
            outputs = model(**inputs)
            hidden_states = outputs.last_hidden_state

        # Find the position of 'bank' token
        bank_position = None
        for j, token in enumerate(tokens):
            if token == 'bank':
                bank_position = j + 1  # +1 because of [CLS] token
                break

        if bank_position is not None:
            # Get embedding for 'bank' token
            embedding = hidden_states[0, bank_position, :]

            print(f"Context {i+1}: '{context}'")
            print(f"  'bank' token embedding shape: {embedding.shape}")
            print(f"  'bank' embedding norm: {embedding.norm().item():.3f}")
            print()
```

The contextualized embeddings reveal the magic of BERT's bidirectional processing:

```text
BERT Contextualized Embeddings:
Context 1: 'The bank of the river is muddy.'
  'bank' token embedding shape: torch.Size([768])
  'bank' embedding norm: 13.907

Context 2: 'I need to go to the bank to withdraw money.'
  'bank' token embedding shape: torch.Size([768])
  'bank' embedding norm: 15.690
```

These results demonstrate how the same word `bank` receives completely different 768-dimensional representations in each context. In the river context, the embedding has a norm of 13.907, while in the financial context, it shows 15.690—but more importantly, these vectors point in different directions in the high-dimensional space. BERT achieves this by allowing the `bank` token to attend to all surrounding words bidirectionally: in the first sentence, it attends to `river` and `muddy`, pulling the representation toward the geographical meaning; in the second, it attends to `withdraw` and `money`, shifting the representation toward the financial meaning.

This dynamic representation is what makes BERT so powerful for downstream tasks. Traditional embeddings would use the same vector for `bank` in both contexts, forcing downstream models to disambiguate the meaning. BERT's contextualized embeddings arrive pre-disambiguated, with the correct meaning already encoded in the representation. This explains why BERT-based models achieve superior performance on tasks requiring fine-grained semantic understanding—the heavy lifting of context interpretation happens within BERT itself.

---

### BERT for Real-World Classification

Let's conclude our exploration by seeing how BERT's bidirectional understanding translates into exceptional performance on real-world classification tasks. BERT's ability to capture nuanced meaning from full context makes it particularly effective for sentiment analysis, where understanding often requires processing entire sentences holistically.

In practice, we rarely use the original BERT model "as is" for downstream tasks. Instead, we use fine-tuned models—versions of BERT that have been further trained on specific datasets tailored to particular tasks, such as sentiment analysis, question answering, or named entity recognition. Fine-tuning allows BERT to adapt its powerful general-purpose language understanding to the nuances of a target task, resulting in much higher accuracy. Specifically, we'll be using the DistilBERT model.

DistilBERT is a distilled, smaller version of BERT that retains most of its performance while being faster and more resource-efficient—making it ideal for real-world applications where speed and memory usage matter. In the example below, we use a DistilBERT model that has been fine-tuned on the Stanford Sentiment Treebank (SST-2) dataset for sentiment analysis:

```python
from transformers import pipeline

def bert_classification():
    """Demonstrate BERT for classification"""
    print("BERT Classification:")

    classifier = pipeline("sentiment-analysis",
                          model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

    test_texts = [
        "I love this movie!",
        "This is terrible.",
        "It's okay."
    ]

    for text in test_texts:
        result = classifier(text)
        print(f"'{text}' -> {result[0]['label']} ({result[0]['score']:.3f})")
```

The classification results demonstrate BERT's performances in sentiment understanding:

```text
BERT Classification:
'I love this movie!' -> POSITIVE (1.000)
'This is terrible.' -> NEGATIVE (1.000)
'It's okay.' -> POSITIVE (1.000)
```

Clear emotional expressions like `I love this movie!` and `This is terrible.` receive perfect confidence scores (1.000) for their respective sentiments. The model's bidirectional nature allows it to process the entire utterance simultaneously—the positive verb `love` combined with the emphatic exclamation mark creates an unambiguous positive signal, while `terrible` with its negative connotation and period punctuation clearly indicates negative sentiment.

Interestingly, the neutral statement `It's okay.` is classified as `POSITIVE` with perfect confidence. This reveals an important characteristic of this particular model: trained on movie reviews from the Stanford Sentiment Treebank (which contains only positive/negative labels), it must force even neutral statements into one of these categories. The model likely interprets `okay` as mildly positive rather than negative, demonstrating how BERT's pre-training on massive text corpora gives it a nuanced understanding of subtle language patterns. This behavior highlights the importance of understanding your model's training data and limitations when deploying BERT for real-world applications.

---

### Conclusion and Next Steps

You've now mastered BERT's revolutionary bidirectional encoder architecture and witnessed firsthand how it transforms language understanding tasks. From WordPiece tokenization that gracefully handles any vocabulary to masked language modeling that leverages full bidirectional context, from dynamic contextualized embeddings that capture meaning in context to confident classification performance, you've explored the key innovations that made BERT a cornerstone of modern NLP. Your deep understanding of transformer fundamentals from previous courses has enabled you to appreciate not just what BERT does, but how and why its architectural choices create such powerful representations.

As we continue our journey through the Hugging Face ecosystem, this foundational knowledge of BERT opens doors to countless applications and variations. The practice exercises ahead will solidify your understanding through hands-on implementation, preparing you to leverage BERT's power for your own NLP challenges. Keep experimenting and exploring—the bidirectional understanding you've learned today will prove invaluable as we dive into even more advanced architectures in upcoming lessons!


## Exploring BERT Tokenizer Fundamentals

Now that you've learned about BERT's encoder architecture, it's time to get hands-on with one of its key components — the tokenizer. Your objective is to complete the `explore_bert_tokenizer()` function to see how BERT breaks down text into tokens.

---

### Your Task

You'll need to:

- Create a BERT tokenizer using the pre-trained model
- Apply tokenization and encoding to different text examples
- Observe how WordPiece tokenization handles various scenarios

The provided code includes test sentences that will help you understand how BERT processes different types of text, from simple phrases to complex terms like "COVID-19." Complete the missing parts and watch how BERT's tokenizer intelligently splits words into meaningful subword units!

---

### Starter Code

```python
from transformers import AutoTokenizer

def explore_bert_tokenizer():
    """Explore BERT's WordPiece tokenization"""
    print("BERT WordPiece Tokenization:")

    # TODO: Create a BERT tokenizer

    # Test tokenization scenarios
    test_texts = ["Hello world", "Tokenization", "COVID-19 pandemic"]

    for text in test_texts:
        # TODO: Get tokens for the current text
        # TODO: Get token IDs for the current text

        print(f"Text: '{text}'")
        print(f"  Tokens: {tokens}")
        print(f"  Token IDs: {token_ids}")
        print()

if __name__ == "__main__":
    explore_bert_tokenizer()
```


---

### Solution

Three TODOs needed code: creating the tokenizer, getting the tokens, and getting the token IDs.

```python
# Create a BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

for text in test_texts:
    tokens = tokenizer.tokenize(text)                                  # WordPiece tokens (strings)
    token_ids = tokenizer.encode(text, add_special_tokens=True)        # IDs, with [CLS] and [SEP]
```

- `AutoTokenizer.from_pretrained("bert-base-uncased")` loads BERT's WordPiece tokenizer. The `uncased` model lowercases the input, so `Hello` becomes `hello`.
- The tokenizer is created once, **before** the loop, so it is not reloaded for every text.
- `tokenizer.tokenize(text)` returns the list of subword strings. Words missing from the vocabulary are split into pieces, and `##` marks a continuation piece.
- `tokenizer.encode(text, add_special_tokens=True)` returns a list of integer IDs, with `[CLS]` (101) added at the start and `[SEP]` (102) at the end. That is why the ID list is two entries longer than the token list.
- Without the tokenizer line, `tokenizer` is undefined and the loop raises a `NameError`. Without the two calls inside the loop, `tokens` and `token_ids` are undefined, so the first `print` raises an `UnboundLocalError`.

When the script runs, it prints each of the three texts with its tokens and token IDs. The lesson above shows the expected output for the same inputs, for example `['token', '##ization']` for `Tokenization`. The first run downloads the tokenizer files, so it needs internet access.


In [ ]:
from transformers import AutoTokenizer

def explore_bert_tokenizer():
    """Explore BERT's WordPiece tokenization"""
    print("BERT WordPiece Tokenization:")

    # Create a BERT tokenizer
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

    # Test tokenization scenarios
    test_texts = ["Hello world", "Tokenization", "COVID-19 pandemic"]

    for text in test_texts:
        # Get tokens for the current text
        tokens = tokenizer.tokenize(text)
        # Get token IDs for the current text
        token_ids = tokenizer.encode(text, add_special_tokens=True)

        print(f"Text: '{text}'")
        print(f"  Tokens: {tokens}")
        print(f"  Token IDs: {token_ids}")
        print()

if __name__ == "__main__":
    explore_bert_tokenizer()


## BERT Fills in the Blanks

## Context Changes Everything in BERT

## Building Your First BERT Classifier